# InsightAid AI — Sentiment Model Training (Objective 1)

Trains and evaluates all three sentiment models (XLM-R, mBERT, TF-IDF +
Logistic Regression) against the real `insightaid_sentiment` dataset
prepared by `01_sentiment_data_preparation.ipynb`
(`dataset_version` recorded in `manifest.json`), across both evaluation
paths (original-language, translate-to-English), using the already-tested
code in `ml/sentiment/src/` — no training logic is duplicated here.

**Before running:** Runtime → Change runtime type → **T4 GPU**. XLM-R/mBERT
will be extremely slow on CPU.

**What this does NOT do:** invent any metric. Every number below comes from
`ml/sentiment/src/metrics.py` run against this model's real predictions on
the real held-out test set. If something fails partway, the cells already
run still produced real results — nothing here back-fills a guess.

In [ ]:
# 1. Locate your prepared dataset in Drive, and the repo for ml/sentiment/src.
import os, sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Adjust this if your prepared dataset folder is somewhere else in Drive.
DATA_DIR = "/content/drive/MyDrive/INSIGHTAID_AI/ml_data/insightaid_sentiment_data"
assert os.path.exists(DATA_DIR), f"Dataset folder not found: {DATA_DIR} — update DATA_DIR above."
print("Dataset folder:", DATA_DIR)
print(os.listdir(DATA_DIR))

In [ ]:
# 2. Get ml/sentiment/src (clone the repo — it's public on GitHub).
REPO_DIR = "/content/insightaid-ai"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/danishalif04/InsightAID-AI.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

sys.path.insert(0, os.path.join(REPO_DIR, "ml", "sentiment"))
print("ml/sentiment/src is now importable.")

In [ ]:
# 3. Install training dependencies (pandas/numpy/scikit-learn are already
# in Colab's base image — only the heavy transformer stack is installed here).
!pip install -q torch transformers datasets accelerate sentencepiece sacremoses huggingface_hub

import torch
print("GPU available:", torch.cuda.is_available(), "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (fix this before training XLM-R/mBERT)")

In [ ]:
# 4. Load the real prepared dataset and confirm which dataset_version this run uses.
import json as _json
import pandas as pd

with open(os.path.join(DATA_DIR, "manifest.json")) as f:
    manifest = _json.load(f)
DATASET_VERSION = manifest["dataset_version"]
print("dataset_version:", DATASET_VERSION, "| rows:", manifest["rows"])

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# The prepared files use 'text'/'label' columns (src/sentiment_prep.py's
# STD_COLS) — NOT 'cleaned_text', which is what the app's own processed_data
# export (src/services/mlExportService.js) uses instead. Both
# train_xlmr.train_transformer() and evaluate.run_full_comparison() accept
# text_column/label_column as parameters for exactly this reason.
TEXT_COL, LABEL_COL = "text", "label"
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
assert TEXT_COL in train_df.columns and LABEL_COL in train_df.columns

In [ ]:
# 5. Train the TF-IDF + Logistic Regression baseline (fast, CPU-only —
# a good sanity check before committing time to the transformer runs).
from src.train_tfidf_baseline import train_tfidf_baseline, predict as tfidf_predict

tfidf_model = train_tfidf_baseline(train_df[TEXT_COL], train_df[LABEL_COL])
print("TF-IDF baseline trained on", len(train_df), "rows.")

In [ ]:
# 6. Train XLM-RoBERTa (primary model, FYP2.md §46). Needs the GPU runtime.
from pathlib import Path
from src.train_xlmr import XLMR_CHECKPOINT, train_transformer, predict as xlmr_predict

xlmr_trained = train_transformer(
    XLMR_CHECKPOINT, train_df, val_df,
    output_dir=Path("/content/xlmr_model"),
    text_column=TEXT_COL, label_column=LABEL_COL,
)
print("XLM-R trained. Saved to", xlmr_trained.output_dir)

In [ ]:
# 7. Train mBERT (comparison model — this project's own methodology
# addition, not required by FYP2.md §46).
from src.train_mbert import train_mbert
from src.train_xlmr import predict as mbert_predict  # same generic predict()

mbert_trained = train_mbert(
    train_df, val_df,
    output_dir=Path("/content/mbert_model"),
    text_column=TEXT_COL, label_column=LABEL_COL,
)
print("mBERT trained. Saved to", mbert_trained.output_dir)

In [ ]:
# 8. Evaluate all three models across both evaluation paths (original-
# language vs. translate-to-English first) on the held-out test set.
from src import evaluate as evaluate_module

RUN_VERSION = pd.Timestamp.utcnow().strftime("v%Y%m%d-%H%M%S")  # this training run's own model_version

predict_fns = {
    "tfidf_lr": lambda texts: tfidf_predict(tfidf_model, texts),
    "xlmr": lambda texts: xlmr_predict(xlmr_trained, texts),
    "mbert": lambda texts: mbert_predict(mbert_trained, texts),
}

rows = evaluate_module.run_full_comparison(
    predict_fns, test_df,
    model_version=RUN_VERSION,
    dataset_version=DATASET_VERSION,
    training_samples=len(train_df),
    validation_samples=len(val_df),
    text_column=TEXT_COL, label_column=LABEL_COL,
    results_dir=Path("/content/results"),
)

for row in rows:
    print(f"{row.model_type:10s} {row.eval_path:22s} acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")

## Optional: evaluate against the hand-labelled gold test set

Only run this once `gold_agreed.csv` exists (built from
`gold_test_set/ANNOTATION_GUIDELINES.md`'s double-annotation + agreement
process — a separate, parallel task, not part of this notebook). Per the
guidelines, this is reported **separately** from the public-dataset test
split above, never blended into it — it's the only part of the evaluation
that covers real Malaysian code-mixed text.

In [ ]:
GOLD_PATH = os.path.join(DATA_DIR, "..", "gold_test_set", "gold_agreed.csv")
if os.path.exists(GOLD_PATH):
    gold_df = pd.read_csv(GOLD_PATH)
    gold_df = gold_df[gold_df["final_label"].isin(["positive", "neutral", "negative"])]  # drop 'exclude' rows
    print("Gold test set:", len(gold_df), "items")

    gold_rows = []
    for model_type, predict_fn in predict_fns.items():
        y_pred = predict_fn(list(gold_df["text"]))
        computed, row = evaluate_module.evaluate_predictions(
            list(gold_df["final_label"]), y_pred, list(gold_df["language"]),
            model_name=f"insightaid-sentiment-{model_type}",
            model_version=f"{RUN_VERSION}-gold",  # distinct from the public test-set run (unique constraint on model_versions)
            model_type=model_type,
            eval_path="original_language",  # gold set has no translate-first variant
            dataset_version=DATASET_VERSION,
            training_samples=len(train_df),
            validation_samples=len(val_df),
            test_samples=len(gold_df),
            notes="Evaluated against the hand-labelled gold_test_set, not the public dataset's own test split.",
        )
        print(f"{model_type:10s} gold_test_set  acc={row.accuracy:.4f} f1={row.f1:.4f} n={row.test_samples}")
        gold_rows.append(row)
    rows = rows + gold_rows
else:
    print("No gold_agreed.csv yet at", GOLD_PATH, "— skipping. Build it per gold_test_set/ANNOTATION_GUIDELINES.md first.")

## Record these results in Supabase

Nothing in this app writes to `model_versions` automatically (by design —
only the Supabase **service role** can insert; see
`supabase/migrations/0004_model_versions.sql`). Copy the SQL printed below
into Supabase Dashboard → SQL Editor → New query → Run, once per row.

In [ ]:
def sql_literal(value):
    if value is None:
        return "NULL"
    if isinstance(value, (dict, list)):
        return "'" + _json.dumps(value).replace("'", "''") + "'::jsonb"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

from dataclasses import asdict

for row in rows:
    d = asdict(row)
    cols = ", ".join(d.keys())
    vals = ", ".join(sql_literal(v) for v in d.values())
    print(f"INSERT INTO model_versions ({cols}) VALUES ({vals});")
    print()

## Save everything back to Drive

So the trained models and metrics JSON survive after this Colab session ends.

In [ ]:
import shutil

RESULTS_DEST = os.path.join(DATA_DIR, "..", "training_runs", RUN_VERSION)
os.makedirs(RESULTS_DEST, exist_ok=True)
if os.path.exists("/content/results"):
    shutil.copytree("/content/results", os.path.join(RESULTS_DEST, "metrics"), dirs_exist_ok=True)
print("Metrics JSON copied to", RESULTS_DEST)
print("(The model checkpoints themselves are large — /content/xlmr_model and /content/mbert_model —")
print(" copy those to Drive too with shutil.copytree if you intend to serve this exact model later.)")